**Author:** Javier Lendínez  
**License:** GNU Affero General Public License v3.0 ([LICENSE](../../LICENSE))

# Geometry, Discretization, and Control Surfaces

In this notebook, I build a tapered wing with sweep, camber, and a flap using the actual `VLM` and `geometry` APIs. I inspect the panel geometry, compare its projections, and analyze how the solution changes with flap deflection.

The example extends the geometry cases covered in my thesis. I retain the method's assumptions: steady, incompressible, inviscid flow over a thin lifting surface, without a separation or viscous-drag model.

## Theory and How the Code Works

A wing planform is described by span, root and tip chords, sweep, dihedral, and airfoil sections. For a single symmetric trapezoid with full span $b$, its declared geometric reference area is $S=b(c_r+c_t)/2$. Sweep and dihedral displace section edges; they do not supply a viscous or thickness-flow model. The implementation uses internally stored coordinates `(y, x, z)`.

The geometry stage creates surface edges; the discretization stage divides them into panels, bound-vortex segments, control points, and normals. Local camber slope enters the flow-tangency boundary condition. The prescribed flap changes points behind the hinge; its `deflection_angle`, sweep, and dihedral are radians, whereas flap positions are fractions of semispan and local chord. Inspecting the actual panel data is important because effective panel counts may differ from requested counts.

**Usage and examples.** Run the cells in order. Define a `plane` dictionary, instantiate `VLM`, and call `calculate_geometry()`, `calculate_discretization()`, or `run_vlm()`. The examples compare neutral and deflected flaps, antisymmetric controls, and winglet cant. Checks of panel counts and response trends verify this implementation, not stall behavior or experimental accuracy. See [Leishman: Lifting-Line and Finite-Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/) for the geometric and lifting-surface context.

In [146]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    current = start
    while current != current.parent:
        if (current / 'src' / 'vlm').is_dir():
            return current
        current = current.parent
    raise RuntimeError('Could not locate the VLMPy repository root (expected src/vlm).')


REPO_ROOT = _find_repo_root(Path.cwd())
SRC_VLM = REPO_ROOT / 'src' / 'vlm'
if str(SRC_VLM) not in sys.path:
    sys.path.insert(0, str(SRC_VLM))

print(f'VLM source path added to sys.path: {SRC_VLM}')

VLM source path added to sys.path: c:\Users\javil\Documents\GitHub\VLMPy\src\vlm


In [147]:
import contextlib
import io

import numpy as np
import plotly.graph_objects as go

from lib.vlm import VLM
from lib.geometry import (
    plot_wing_geometry_2d,
    plot_wing_discretization_2d,
    plot_wing_discretization_3d,
)
from lib.vlm import plot_wing_heatmap_2d

## 1. Tapered Wing with Camber and Flap

In this configuration, `sweep` and `dihedral` are specified in radians; `flap_start` and `flap_end` delimit fractions of the semispan; and `flap_hinge_chord` gives the hinge location as a fraction of the local chord.

For `deflection_type='symmetrical'`, positive deflection moves the trailing edge downward, according to the geometric transformation in `generate_panels_for_surface`. I interpret the lift increase as a trend for this case and this kinematic model, not as a prediction of separation or maximum flap load.

In [148]:
plane = {
    'wing_sections': [
        {
            'chord_root': 1.2,
            'chord_tip': 0.6,
            'span_fraction': 8.0,          # total span [m]
            'sweep': np.radians(20.0),     # leading-edge sweep [rad]
            'dihedral': np.radians(4.0),
            'NACA_root': '4412',
            'NACA_tip': '2412',
            'flap_start': 0.55,            # fraction of half-span
            'flap_end': 0.90,
            'flap_hinge_chord': 0.75,       # hinge at 75% local chord
            'deflection_angle': np.radians(15.0),
            'deflection_type': 'symmetrical',
        }
    ]
}

vlm = VLM(plane=plane, u=45.0, rho=1.225, alpha=np.radians(4.0), beta=0.0, n=24, m=8)

## 2. Geometry and Discretization by Stage

I run geometry calculation and panel generation separately to inspect the cost and cache indicators for each stage. This separation also helps identify which stage first shows an inconsistency between geometry and mesh.

In [149]:
_out = io.StringIO()
with contextlib.redirect_stdout(_out):
    vlm.calculate_geometry()
    vlm.calculate_discretization()

print(f'Wing span (this section): {vlm.total_wing_span:.2f} m')
print(f'Panels: n={vlm.n}, m={vlm.m}, total wing panels={vlm.n * vlm.m}')
print(f'Wing planform area: {vlm.wing_area:.4f} m^2')
print('Timings [s]:', {k: v for k, v in vlm.timings.items() if not k.endswith('_hit')})
print('Cache-hit flags:', {k: v for k, v in vlm.timings.items() if k.endswith('_hit')})

Wing span (this section): 8.00 m
Panels: n=24, m=8, total wing panels=192
Wing planform area: 7.2302 m^2
Timings [s]: {'geometry': 0.0005181999877095222, 'discretization': 0.019938900019042194}
Cache-hit flags: {'geometry_cache_hit': True, 'discretization_cache_hit': False}


### Verification: Geometric Quarter-Chord Location

At each section endpoint, the geometric quarter chord must lie one quarter of the distance from the leading edge to the trailing edge. This relation is invariant under rigid rotation and translation. The diagnostic below reports any discrepancy rather than hiding it. For tapered wings, a discrepancy at the root also matters because this location is used as the local twist axis during interpolation; its effect on twisted meshes and aerodynamic results needs a separate controlled study.

In [150]:
leading_edge = np.asarray(vlm.wing_geometry['leading_edge'], dtype=float).T
trailing_edge = np.asarray(vlm.wing_geometry['trailing_edge'], dtype=float).T
quarter_chord = np.asarray(vlm.wing_geometry['quarter_chord'], dtype=float).T
expected_quarter_chord = leading_edge + 0.25 * (trailing_edge - leading_edge)
quarter_chord_error = np.linalg.norm(quarter_chord - expected_quarter_chord, axis=1)
assert leading_edge.shape == trailing_edge.shape == quarter_chord.shape
assert np.isfinite(quarter_chord_error).all()
print(f'Maximum quarter-chord position discrepancy: {quarter_chord_error.max():.6f} m')
if quarter_chord_error.max() > 1e-10:
    worst_endpoint = int(np.argmax(quarter_chord_error))
    print(f'INVESTIGATE endpoint {worst_endpoint}: stored={quarter_chord[worst_endpoint]}, geometric={expected_quarter_chord[worst_endpoint]}')
else:
    print('PASS: all quarter-chord endpoints agree with the section geometry.')

Maximum quarter-chord position discrepancy: 0.150000 m
INVESTIGATE endpoint 1: stored=[ 0.14963461 -0.         -0.01046347], geometric=[ 0.29926922  0.         -0.02092694]


### Verification: Geometric Quarter-Chord Location

For each untwisted section endpoint, the geometric quarter chord must be the leading edge plus one quarter of the vector from leading to trailing edge. This relation remains valid after a rigid rotation and translation. The following diagnostic reports any discrepancy rather than masking it: the current implementation initializes the root quarter chord with the tip chord for a tapered first section. That reference is also used as the local twist axis during interpolation; its effect on twisted meshes and aerodynamic results requires a separate controlled study.

## 3. Panel Structure

Each element of `vlm.panel_data` contains the index, corners, quarter-chord line nodes, control point, and auxiliary normal data. Coordinates follow the internal `(y, x, z)` convention: spanwise, chordwise, and vertical. Inspecting this record connects solver quantities to the geometry being plotted.

In [151]:
index, corners, quarter_chord_nodes, control_point, normal_info = vlm.panel_data[0]
normal_vector, three_quarter_midpoint = normal_info

print(f'Panel index: {index}')
print('Corner coordinates (y, x, z):')
for corner in corners:
    print(f'  {corner}')
print(f'Quarter-chord (bound vortex) nodes: {quarter_chord_nodes}')
print(f'Control point (3/4 chord): {control_point}')
print(f'Panel normal vector: {normal_vector}')

Panel index: 1
Corner coordinates (y, x, z):
  [-3.9902562   1.46826053  0.17703647]
  [-3.9902562   1.54307784  0.17180473]
  [-3.65773485  1.42695757  0.15661572]
  [-3.65773485  1.34590549  0.16228343]
Quarter-chord (bound vortex) nodes: [[-3.990256201039297, 1.4869648589240536, 0.17572853571745084], [-3.657734850952689, 1.3661685083374444, 0.16086650209387954]]
Control point (3/4 chord): [-3.8239955259959926, 1.4655340293440235, 0.1655726566500353]
Panel normal vector: [0.06975647 0.06958655 0.99513403]


## 4. Camber Slope $dz/dx$

`vlm.dz_c` stores the local slope associated with the panels and contributes to the boundary condition. I check that there is one entry per panel and summarize its range. This check verifies the correspondence between geometry data and panels; it is not, by itself, a comparison with an experimental airfoil profile.

In [152]:
print(f'dz/dx entries: {len(vlm.dz_c)} (panel count: {len(vlm.panel_data)})')
assert len(vlm.dz_c) == len(vlm.panel_data)
print(f'Camber slope range: [{min(vlm.dz_c):.4f}, {max(vlm.dz_c):.4f}]')

dz/dx entries: 192 (panel count: 192)
Camber slope range: [-0.1687, 0.1194]


### Panel-Area and Camber-Slope Diagnostic

The next plot relates each control point's spanwise coordinate to the corresponding numerical panel area and camber slope. It helps identify abrupt mesh changes or ordering mistakes; the colors are not pressure coefficients or measured surface geometry.

In [153]:
panel_span = np.asarray([panel[3][0] for panel in vlm.panel_data], dtype=float)
panel_area = np.asarray(vlm.panel_areas, dtype=float)
camber_slope = np.asarray(vlm.dz_c, dtype=float)
assert panel_span.shape == panel_area.shape == camber_slope.shape
assert np.isfinite(panel_area).all() and np.all(panel_area > 0)
fig_panel_quality = go.Figure(go.Scatter(
    x=panel_span, y=panel_area, mode='markers',
    marker=dict(color=camber_slope, colorscale='RdBu', colorbar=dict(title='dz/dx')),
    name='Panels',
))
fig_panel_quality.update_layout(
    title='Panel area and camber slope across the span',
    xaxis_title='Control-point span coordinate y [m]', yaxis_title='Panel area [m²]',
    template='plotly_white',
)
fig_panel_quality.show()

In [154]:
fig = go.Figure()
plot_wing_heatmap_2d(fig, vlm.panel_data, vlm.dz_c, title='Curvature Distribution', legend='Curvature [1/m]', u_vec=getattr(vlm, 'u_', None))
fig.show()

## 5. Geometry and Mesh Views

The three views distinguish the planform, side projection, and spatial panel layout. The 2D view helps inspect edges and quarter-chord lines; the 3D view can reveal orientation errors that may be missed in a flat projection.

In [155]:
fig_planform = go.Figure()
fig_planform = plot_wing_geometry_2d(fig_planform, vlm.wing_geometry, legend='Wing Geometry')
fig_planform.show()

In [156]:
fig_panels = go.Figure()
fig_panels = plot_wing_discretization_2d(fig_panels, vlm.panel_data)
fig_panels.show()

Execution time: 0.03 seconds


In [157]:
fig_3d = go.Figure()
fig_3d = plot_wing_discretization_3d(fig_3d, vlm.panel_data)
fig_3d.show()

Execution time: 0.03 seconds


## 6. Neutral and Deflected-Flap Comparison

The planform, side, and 3D views make it possible to inspect the panel arrangement from different directions. Reviewing all three helps catch geometric details that are difficult to see in a single projection. The following calculation compares the neutral and deflected configurations at the same flight condition.

In [158]:
_out = io.StringIO()
with contextlib.redirect_stdout(_out):
    vlm.run_vlm()
cl_deflected = vlm.CL
cd_deflected = vlm.CD

plane_neutral = {'wing_sections': [dict(plane['wing_sections'][0])]}
plane_neutral['wing_sections'][0]['deflection_angle'] = 0.0

vlm_neutral = VLM(plane=plane_neutral, u=45.0, rho=1.225, alpha=np.radians(4.0), beta=0.0, n=24, m=8)
_out = io.StringIO()
with contextlib.redirect_stdout(_out):
    vlm_neutral.run_vlm()

deflection_deg = np.degrees(plane['wing_sections'][0]['deflection_angle'])
print(f'Flap neutral   : CL={vlm_neutral.CL:.4f}, CD={vlm_neutral.CD:.5f}')
print(f'Flap deflected : CL={cl_deflected:.4f}, CD={cd_deflected:.5f}')
print(f'Delta CL from a {deflection_deg:.0f} deg deflection: {cl_deflected - vlm_neutral.CL:+.4f}')

assert cl_deflected > vlm_neutral.CL, 'A positive (trailing-edge-down) flap deflection should increase CL at fixed alpha.'

Flap neutral   : CL=0.6168, CD=0.00840
Flap deflected : CL=0.8280, CD=0.02213
Delta CL from a 15 deg deflection: +0.2112


## 7. Limitations

- The flap is represented by a rigid transformation of points downstream of the hinge; hinge moment and a physical discontinuity between surfaces are not modeled.
- The model does not include viscosity, separation, or a deflection limit associated with loss of flap effectiveness. Therefore, results at 20° describe only the mathematical response of the linear VLM.
- The response depends on discretization; quantifying sensitivity would require repeating the sweep on more than one mesh.

## 8. Flap-Deflection Sweep

I keep the wing, angle of attack, and flight conditions fixed. I vary only the flap deflection and plot $C_L$ and $C_D$ separately so their different scales do not hide either trend. The cell also checks that all cases retain the effective mesh `(12, 6)`.

On this mesh, increasing deflection from 0° to 20° raises $C_L$ from 0.61390 to 0.87112 and induced $C_D$ from 0.008078 to 0.025491. The trend is monotonic over this interval. At 15°, the `(12, 6)` mesh sweep gives $C_L=0.80813$, compared with 0.8280 in the neutral/deflected comparison using `(24, 8)`. This difference is not a physical contradiction: the calculations use different resolutions.

In [159]:
import contextlib
import io
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

current = Path.cwd()
while current != current.parent and not (current / 'src' / 'vlm').is_dir():
    current = current.parent
if not (current / 'src' / 'vlm').is_dir():
    raise RuntimeError('Could not locate the VLMPy repository root.')
sys.path.insert(0, str(current / 'src' / 'vlm'))

section_sweep = {
    'chord_root': 1.2, 'chord_tip': 0.6, 'span_fraction': 8.0,
    'sweep': np.radians(20.0), 'dihedral': np.radians(4.0),
    'NACA_root': '4412', 'NACA_tip': '2412',
    'flap_start': 0.55, 'flap_end': 0.90,
    'flap_hinge_chord': 0.75, 'deflection_type': 'symmetrical',
}
deflections_deg = [0.0, 5.0, 10.0, 15.0, 20.0]
cl_by_deflection = []
cd_by_deflection = []

for deflection_deg in deflections_deg:
    section_case = dict(section_sweep)
    section_case['deflection_angle'] = np.radians(deflection_deg)
    plane_case = {'wing_sections': [section_case]}
    case = VLM(plane_case, 45.0, 1.225, np.radians(4.0), 0.0, 12, 6)
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    assert (case.n, case.m) == (12, 6), 'The solver changed the requested mesh.'
    cl_by_deflection.append(float(case.CL))
    cd_by_deflection.append(float(case.CD))
    print(
        'Deflection={:>4.0f}°; effective mesh=({},{}): CL={:.5f}, CD={:.6f}'.format(
            deflection_deg, case.n, case.m, case.CL, case.CD
        )
    )

cl_figure = go.Figure()
cl_figure.add_trace(go.Scatter(
    x=deflections_deg, y=cl_by_deflection,
    mode='lines+markers', name='CL',
))
cl_figure.update_layout(
    title='Effect of flap deflection on CL',
    xaxis_title='Flap deflection [deg]',
    yaxis_title='CL [-]', template='plotly_white',
)
cl_figure.show()

cd_figure = go.Figure()
cd_figure.add_trace(go.Scatter(
    x=deflections_deg, y=cd_by_deflection,
    mode='lines+markers', name='Induced CD',
))
cd_figure.update_layout(
    title='Effect of flap deflection on induced CD',
    xaxis_title='Flap deflection [deg]',
    yaxis_title='Induced CD [-]', template='plotly_white',
)
cd_figure.show()

assert np.isfinite(cl_by_deflection).all()
assert np.isfinite(cd_by_deflection).all()

Deflection=   0°; effective mesh=(12,6): CL=0.61390, CD=0.008078
Deflection=   5°; effective mesh=(12,6): CL=0.67935, CD=0.010804
Deflection=  10°; effective mesh=(12,6): CL=0.74416, CD=0.014674
Deflection=  15°; effective mesh=(12,6): CL=0.80813, CD=0.019604
Deflection=  20°; effective mesh=(12,6): CL=0.87112, CD=0.025491


## 6. Neutral and Deflected Flap Comparison

I solve the same geometry and flight conditions with the flap deflected and with zero deflection. The difference in $C_L$ quantifies the model response for this case; it should not be attributed to viscous effects, which are not included.

In [160]:
aileron_section = {
    'chord_root': 1.0, 'chord_tip': 1.0, 'span_fraction': 10.0,
    'sweep': 0.0, 'dihedral': 0.0,
    'NACA_root': '0106', 'NACA_tip': '0106',
    'flap_start': 0.5, 'flap_end': 1.0,
    'flap_hinge_chord': 0.75, 'deflection_type': 'antisymmetrical',
}
aileron_deflections_deg = [-12.0, 0.0, 12.0]
aileron_cl_global = []
aileron_cd_induced = []
aileron_cl_local = []
aileron_y = None

for deflection_deg in aileron_deflections_deg:
    section_case = dict(aileron_section)
    section_case['deflection_angle'] = np.radians(deflection_deg)
    case = VLM(
        {'wing_sections': [section_case]}, 50.0, 1.225,
        0.0, 0.0, 24, 8,
    )
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    assert (case.n, case.m) == (24, 8)
    y_positions = np.array([
        np.mean([
            panel[3][0]
            for panel in case.panel_data[i * case.m:(i + 1) * case.m]
        ])
        for i in range(case.n)
    ])
    if aileron_y is None:
        aileron_y = y_positions
    else:
        np.testing.assert_allclose(y_positions, aileron_y, atol=1e-12)
    aileron_cl_global.append(float(case.CL))
    aileron_cd_induced.append(float(case.CD))
    aileron_cl_local.append(np.asarray(case.CL_locals_wing, dtype=float))
    left_lift = float(np.sum(np.asarray(case.lift_wing)[y_positions < 0.0]))
    right_lift = float(np.sum(np.asarray(case.lift_wing)[y_positions > 0.0]))
    print(
        'Deflection={:+.0f}°: CL={:.6f}, induced CD={:.6f}, left/right lift={:.2f}/{:.2f} N'.format(
            deflection_deg, case.CL, case.CD, left_lift, right_lift
        )
    )

zero_index = aileron_deflections_deg.index(0.0)
positive_index = aileron_deflections_deg.index(12.0)
negative_index = aileron_deflections_deg.index(-12.0)
assert abs(aileron_cl_global[zero_index]) < 1e-8
assert aileron_cd_induced[positive_index] > aileron_cd_induced[zero_index]
np.testing.assert_allclose(
    aileron_cl_local[positive_index], aileron_cl_local[negative_index][::-1],
    rtol=2e-5, atol=5e-6,
 )
np.testing.assert_allclose(
    aileron_cd_induced[positive_index], aileron_cd_induced[negative_index],
    rtol=1e-8, atol=1e-10,
 )

aileron_figure = go.Figure()
for deflection_deg, local_cl in zip(aileron_deflections_deg, aileron_cl_local):
    aileron_figure.add_trace(go.Scatter(
        x=aileron_y, y=local_cl, mode='lines+markers',
        name='Deflection={:+.0f}°'.format(deflection_deg),
    ))
aileron_figure.update_layout(
    title='Spanwise CL distribution with antisymmetric ailerons',
    xaxis_title='Spanwise coordinate y [m]',
    yaxis_title='Local CL [-]', template='plotly_white',
 )
aileron_figure.show()

aileron_cd_figure = go.Figure()
aileron_cd_figure.add_trace(go.Scatter(
    x=aileron_deflections_deg, y=aileron_cd_induced,
    mode='lines+markers', name='Induced CD',
))
aileron_cd_figure.update_layout(
    title='Induced CD versus antisymmetric deflection',
    xaxis_title='Deflection on the +y semispan [deg]',
    yaxis_title='Induced CD [-]', template='plotly_white',
 )
aileron_cd_figure.show()

assert np.isfinite(aileron_cl_global).all()
assert np.isfinite(aileron_cd_induced).all()

Deflection=-12°: CL=0.000000, induced CD=0.009964, left/right lift=2100.96/-2100.96 N
Deflection=+0°: CL=0.000000, induced CD=0.000000, left/right lift=0.00/0.00 N
Deflection=+12°: CL=-0.000000, induced CD=0.009964, left/right lift=-2100.96/2100.96 N


## 10. Winglets

I use a symmetric wing with a 10 m span, 1 m chord, NACA 4412, 30° sweep, and 5° dihedral at $\alpha=5^\circ$, $V=50\,\mathrm{m/s}$, and $\rho=1.225\,\mathrm{kg/m^3}$. The winglet has a 1.0 m root chord, matching the wing-tip chord at the section junction, and a 0.05 m tip chord, with NACA 4412/0012 sections. Its `span_fraction` is 1.0 m total, halved by the symmetric geometry, so the vertical height per tip at 75° is $0.5\sin(75^\circ)\approx0.483$ m.

Winglets can reduce **induced** drag when their geometry improves the spanwise load distribution at the same required lift; a reduction is not guaranteed for every size, cant, or operating condition. Added wetted area can also increase total drag, which this inviscid VLM does not model. The first sweep below holds $\alpha=5^\circ$ fixed, so its `CL` changes with cant and its `CD` values are not an equal-lift comparison. 

In [161]:
winglet_base_section = {
    'chord_root': 1.0, 'chord_tip': 1.0, 'span_fraction': 10.0,
    'sweep': np.radians(20.0), 'dihedral': np.radians(5.0),
    'NACA_root': '4412', 'NACA_tip': '4412',
}

winglet_base_conditions = (50.0, 1.225, np.radians(5.0), 0.0, 100, 20)
winglet_cant_angles_deg = np.array([5, 15, 25, 35, 45, 55, 65, 75])
winglet_cl = []
winglet_cdi = []

base_case = VLM(
    {'wing_sections': [winglet_base_section]}, *winglet_base_conditions,
)
captured = io.StringIO()
with contextlib.redirect_stdout(captured):
    base_case.run_vlm()
base_cl = float(base_case.CL)
base_cdi = float(base_case.CD)
print('Baseline wing (alpha=5°): CL={:.6f}, CDi={:.6f}'.format(base_cl, base_cdi))

for cant_deg in winglet_cant_angles_deg:
    winglet_section = {
        'chord_root': winglet_base_section['chord_root'], 'chord_tip': 0.05,
        'span_fraction': 1.0, 'sweep': np.radians(80.0),
        'dihedral': np.radians(float(cant_deg)),
        'NACA_root': '4412', 'NACA_tip': '0012',
    }
    assert winglet_section['chord_root'] == winglet_base_section['chord_root']
    plane_sections = [winglet_base_section, winglet_section]
    total_section_span = sum(section['span_fraction'] for section in plane_sections)
    requested_n = round(winglet_base_conditions[4] * total_section_span / winglet_base_section['span_fraction'])
    case = VLM(
        {'wing_sections': plane_sections},
        *winglet_base_conditions[:4], requested_n, winglet_base_conditions[5],
    )
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    assert case.n == requested_n
    assert case.m == winglet_base_conditions[5]
    assert np.isfinite(case.gammas).all()
    winglet_cl.append(float(case.CL))
    winglet_cdi.append(float(case.CD))
    vertical_height = 0.5 * np.sin(np.radians(float(cant_deg)))
    print(
        'Cant angle={:2.0f}°; effective mesh=({},{}); vertical height per tip={:.3f} m; CL={:.6f}; CDi={:.6f}'.format(
            cant_deg, case.n, case.m, vertical_height, case.CL, case.CD
        )
    )

assert np.isfinite(winglet_cl).all()
assert np.isfinite(winglet_cdi).all()
winglet_cl_figure = go.Figure()
winglet_cl_figure.add_trace(go.Scatter(
    x=winglet_cant_angles_deg, y=winglet_cl,
    mode='lines+markers', name='With winglets',
))
winglet_cl_figure.add_hline(
    y=base_cl, line_dash='dash', annotation_text='Baseline wing at alpha=5°',
)
winglet_cl_figure.update_layout(
    title='CL versus winglet cant angle at fixed alpha=5°',
    xaxis_title='Winglet cant angle [deg]',
    yaxis_title='CL [-]', template='plotly_white',
)
winglet_cl_figure.show()

winglet_cdi_figure = go.Figure()
winglet_cdi_figure.add_trace(go.Scatter(
    x=winglet_cant_angles_deg, y=winglet_cdi,
    mode='lines+markers', name='With winglets',
))
winglet_cdi_figure.add_hline(
    y=base_cdi, line_dash='dash', annotation_text='Baseline wing at alpha=5°',
)
winglet_cdi_figure.update_layout(
    title='Induced CD versus winglet cant angle at fixed alpha=5°',
    xaxis_title='Winglet cant angle [deg]',
    yaxis_title='Induced CD [-]', template='plotly_white',
)
winglet_cdi_figure.show()

print(
    'Fixed-alpha change in CDi versus baseline wing: min={:+.1f} %, max={:+.1f} %'.format(
        100.0 * (np.min(winglet_cdi) / base_cdi - 1.0),
        100.0 * (np.max(winglet_cdi) / base_cdi - 1.0),
    )
)

fig_3d = go.Figure()
fig_3d = plot_wing_discretization_3d(fig_3d, base_case.panel_data)
fig_3d.show()

fig_3d = go.Figure()
fig_3d = plot_wing_discretization_3d(fig_3d, case.panel_data)
fig_3d.show()

Baseline wing (alpha=5°): CL=0.783921, CDi=0.015135
Cant angle= 5°; effective mesh=(110,20); vertical height per tip=0.044 m; CL=0.837275; CDi=-0.002640
Cant angle=15°; effective mesh=(110,20); vertical height per tip=0.129 m; CL=0.838106; CDi=-0.002832
Cant angle=25°; effective mesh=(110,20); vertical height per tip=0.211 m; CL=0.837656; CDi=-0.002675
Cant angle=35°; effective mesh=(110,20); vertical height per tip=0.287 m; CL=0.836560; CDi=-0.002342
Cant angle=45°; effective mesh=(110,20); vertical height per tip=0.354 m; CL=0.834441; CDi=-0.000967
Cant angle=55°; effective mesh=(110,20); vertical height per tip=0.410 m; CL=0.829542; CDi=0.001395
Cant angle=65°; effective mesh=(110,20); vertical height per tip=0.453 m; CL=0.820698; CDi=0.005356
Cant angle=75°; effective mesh=(110,20); vertical height per tip=0.483 m; CL=0.807042; CDi=0.010102


Fixed-alpha change in CDi versus baseline wing: min=-118.7 %, max=-33.3 %
Execution time: 0.23 seconds


Execution time: 0.27 seconds


In [162]:
def solve_winglet_forces(sections, alpha_deg):
    total_section_span = sum(section['span_fraction'] for section in sections)
    requested_n = round(100 * total_section_span / 10.0)
    case = VLM(
        {'wing_sections': sections},
        50.0, 1.225, np.radians(alpha_deg), 0.0, requested_n, 20,
    )
    with contextlib.redirect_stdout(io.StringIO()):
        case.run_vlm()
    force = np.asarray(case.aerodynamic_coefficients['force'], dtype=float)
    velocity = np.asarray([case.u_[1], case.u_[0], case.u_[2]], dtype=float)
    velocity_direction = velocity / np.linalg.norm(velocity)
    drag_force = -float(np.dot(force, velocity_direction))
    vertical_force = float(force[2])
    return case, vertical_force, drag_force


reference_case, reference_vertical_force, reference_drag_force = solve_winglet_forces(
    [winglet_base_section], 5.0,
)
matched_drag_forces = []
matched_alpha_deg = []
matched_vertical_forces = []

for cant_deg in winglet_cant_angles_deg:
    winglet_section = {
        'chord_root': winglet_base_section['chord_tip'], 'chord_tip': 0.05,
        'span_fraction': 1.0, 'sweep': 0.0,
        'dihedral': np.radians(float(cant_deg)),
        'NACA_root': '4412', 'NACA_tip': '0012',
    }
    assert winglet_section['chord_root'] == winglet_base_section['chord_tip']
    sections = [winglet_base_section, winglet_section]
    _, vertical_low, _ = solve_winglet_forces(sections, 3.0)
    _, vertical_high, _ = solve_winglet_forces(sections, 7.0)
    assert (vertical_low - reference_vertical_force) * (vertical_high - reference_vertical_force) <= 0
    vertical_slope = (vertical_high - vertical_low) / 4.0
    assert abs(vertical_slope) > 1e-8
    alpha_match = 3.0 + (
        (reference_vertical_force - vertical_low) / vertical_slope
    )
    for _ in range(3):
        _, vertical_match, drag_match = solve_winglet_forces(sections, alpha_match)
        force_error = reference_vertical_force - vertical_match
        if abs(force_error) < 1e-3:
            break
        alpha_match += force_error / vertical_slope
    assert abs(vertical_match - reference_vertical_force) < 1e-3
    matched_alpha_deg.append(alpha_match)
    matched_vertical_forces.append(vertical_match)
    matched_drag_forces.append(drag_match)
    print(
        'Cant={:2.0f}°; alpha={:.3f}°; Fz={:.3f} N; D_i={:.6f} N ({:+.2f} %)'.format(
            cant_deg, alpha_match, vertical_match, drag_match,
            100.0 * (drag_match / reference_drag_force - 1.0),
        )
    )

matched_drag_figure = go.Figure()
matched_drag_figure.add_trace(go.Scatter(
    x=winglet_cant_angles_deg,
    y=matched_drag_forces,
    mode='lines+markers',
    name='Wing with winglets',
))
matched_drag_figure.add_hline(
    y=reference_drag_force,
    line_dash='dash',
    annotation_text='Baseline at same Fz (alpha=5°)',
)
matched_drag_figure.update_layout(
    title='Induced drag at matched vertical force (baseline alpha=5°)',
    xaxis_title='Winglet cant angle [deg]',
    yaxis_title='Freestream-axis induced force [N]',
    template='plotly_white',
)
matched_drag_figure.show()

assert np.isfinite(matched_drag_forces).all()
assert np.isfinite(matched_alpha_deg).all()

fig_3d = go.Figure()
fig_3d = plot_wing_discretization_3d(fig_3d, reference_case.panel_data)
fig_3d.show()

fig_3d = go.Figure()
fig_3d = plot_wing_discretization_3d(fig_3d, case.panel_data)
fig_3d.show()

Cant= 5°; alpha=4.332°; Fz=11241.942 N; D_i=236.175812 N (+1.91 %)
Cant=15°; alpha=4.322°; Fz=11241.942 N; D_i=234.201269 N (+1.05 %)
Cant=25°; alpha=4.332°; Fz=11241.942 N; D_i=231.311898 N (-0.19 %)
Cant=35°; alpha=4.344°; Fz=11241.942 N; D_i=232.722404 N (+0.42 %)
Cant=45°; alpha=4.392°; Fz=11241.942 N; D_i=230.772800 N (-0.42 %)
Cant=55°; alpha=4.445°; Fz=11241.942 N; D_i=232.043985 N (+0.12 %)
Cant=65°; alpha=4.538°; Fz=11241.942 N; D_i=229.583489 N (-0.94 %)
Cant=75°; alpha=4.651°; Fz=11241.942 N; D_i=228.965192 N (-1.20 %)


Execution time: 0.21 seconds


Execution time: 0.23 seconds


### Results and Limitations

At fixed $\alpha=5^\circ$, the baseline gives $C_L=0.777709$ and $C_{D_i}=0.012924$. With spanwise panel density held constant (100 panels for the 10 m baseline and 110 for the 11 m wing-plus-winglet), $C_{D_i}$ remains above the baseline for all tested cant angles: +4.3% at 75° to +18.8% at 5°. This is not an equal-lift comparison because $C_L$ also changes with cant.

The force-matched comparison targets the baseline vertical force, $F_z=10283.735$ N. Using the solver's vector force output projected onto the freestream, the baseline induced-force estimate is 197.892 N; the winglet cases range from 199.791 N at 75° (+0.96%) to 212.227 N at 5° (+7.24%). The winglet angle of attack is adjusted from 4.330° to 4.655° to match $F_z$. Thus, this particular geometry does not reduce the solver's estimated induced drag, even at matched lift.

This result does **not** show that winglets generally increase induced drag: the tested winglet is not optimized, and the theory only predicts a benefit for suitable geometry and loading at comparable lift. The force-matched plot is an internal check because it uses the same VLM wake and induced-velocity model, not an independent method. This environment has no AVL, XFLR5, or VSPAERO installation; an external comparison at matched lift and consistent reference area is still needed before treating the trend as validated. The [TFE](../Thesis/TFE_JavierLendinezCastillo_143219.pdf) has no winglet-specific case to use as a numerical reference.

## Verification: Separate Flap-Mesh Refinements

The neutral and $15^\circ$ flap cases are recomputed at identical flight conditions. The first series doubles the requested spanwise count with fixed chordwise count; the second increases the chordwise count at fixed spanwise count. The table reports the **effective** meshes used for both cases and $\Delta C_L$ and $\Delta C_D$ relative to the neutral case on the same mesh. This is a discretization study, not experimental validation. In particular, changing the panel-grid alignment with the flap hinge or spanwise ends can make the control response non-monotonic; no convergence is assumed or tolerance adjusted to conceal that behavior.

In [163]:
flap_mesh_studies = {
    'spanwise': [(12, 6), (24, 6), (48, 6)],
    'chordwise': [(24, 4), (24, 8), (24, 12)],
}
flap_mesh_results = {}
for direction, meshes in flap_mesh_studies.items():
    rows = []
    for requested_n, requested_m in meshes:
        solutions = []
        for deflection in (0.0, 15.0):
            section = dict(section_sweep, deflection_angle=np.deg2rad(deflection))
            wing_case = VLM({'wing_sections': [section]}, 45.0, 1.225, np.deg2rad(4.0), 0.0, requested_n, requested_m)
            with contextlib.redirect_stdout(io.StringIO()):
                wing_case.run_vlm()
            solutions.append(wing_case)
        neutral, deflected = solutions
        assert (neutral.n, neutral.m) == (deflected.n, deflected.m)
        assert np.isfinite([neutral.CL, neutral.CD, deflected.CL, deflected.CD]).all()
        rows.append({'requested': (requested_n, requested_m), 'effective': (neutral.n, neutral.m),
                     'delta_cl': float(deflected.CL - neutral.CL), 'delta_cd': float(deflected.CD - neutral.CD)})
    flap_mesh_results[direction] = rows
    print(f'\n{direction.capitalize()} refinement:')
    for row in rows:
        print(f"requested={row['requested']}, effective={row['effective']}, ΔCL={row['delta_cl']:+.6f}, ΔCD={row['delta_cd']:+.6f}")
    differences = np.diff([row['delta_cl'] for row in rows])
    print('Successive ΔCL changes:', differences)
    print('Monotonic sequence:', bool(differences[0] * differences[1] > 0))

flap_mesh_figure = go.Figure()
for direction, rows in flap_mesh_results.items():
    axis = 0 if direction == 'spanwise' else 1
    flap_mesh_figure.add_trace(go.Scatter(x=[row['effective'][axis] for row in rows],
        y=[row['delta_cl'] for row in rows], mode='lines+markers', name=direction))
flap_mesh_figure.update_layout(title='Flap lift increment versus mesh resolution',
    xaxis_title='Effective panels in refined direction', yaxis_title='ΔCL: 15° minus 0°', template='plotly_white')
flap_mesh_figure.show()


Spanwise refinement:
requested=(12, 6), effective=(12, 6), ΔCL=+0.194231, ΔCD=+0.011526
requested=(24, 6), effective=(24, 6), ΔCL=+0.203465, ΔCD=+0.013105
requested=(48, 6), effective=(48, 6), ΔCL=+0.197589, ΔCD=+0.013779
Successive ΔCL changes: [ 0.00923425 -0.00587557]
Monotonic sequence: False

Chordwise refinement:
requested=(24, 4), effective=(24, 4), ΔCL=+0.200826, ΔCD=+0.013104
requested=(24, 8), effective=(24, 8), ΔCL=+0.211246, ΔCD=+0.013734
requested=(24, 12), effective=(24, 12), ΔCL=+0.215057, ΔCD=+0.013950
Successive ΔCL changes: [0.01041977 0.00381081]
Monotonic sequence: True


## Verification and Validation

Panel-array dimensions, finite geometry, mirrored controls, and observed force trends are implementation checks. The thesis comparisons and mesh studies have their stated assumptions; they do not establish experimental accuracy or a stall limit.

## References

- [Leishman: Lifting-Line and Finite-Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/).
- [VLMPy engineering guidance](../README-agent.md) for units, geometry, and verification discipline.

## Use Cases

Use the geometry and mesh views to inspect taper, sweep, flap placement, and cant before comparing aerodynamic responses.

## Possible Improvements

Add separate chordwise and spanwise convergence studies and compare the generated geometry and loads with a documented independent case, including units and control-sign conventions.

## Next Steps

Examine surface coupling in [Multi-Surface Configuration](03_multi_surface_configuration.ipynb), and use [Static Aerodynamic Derivatives](07_static_aerodynamic_derivatives.ipynb) to study local sensitivities.